<a href="https://colab.research.google.com/github/meenakshishibu16/freshcart-pipelines/blob/main/iris_project/iris_mage_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part C: Mage Iris pipeline (Colab)
Runs the project that is already in the Git repo: `load_iris → validate_iris → prepare_data → train_model → (save_results, evaluate_model)`.
Run top to bottom and keep the outputs visible (they are part of the submission).

## 0. Get the code from GitHub

In [ ]:
GITHUB_USER = "meenakshishibu16"          # <- your GitHub user name
REPO = "freshcart-pipelines"
!git clone https://github.com/{GITHUB_USER}/{REPO}.git /content/{REPO}
%cd /content/{REPO}
!git log --oneline | head

Cloning into '/content/freshcart-pipelines'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 61 (delta 16), reused 58 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (61/61), 14.23 KiB | 2.03 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/freshcart-pipelines
1b35c9a Add validate_iris block with @test and the Mage Colab notebook
3299bfc Add evaluate_model branch after train_model
7107f31 Fix test_size back to 0.2
326e79a Change test_size to 0.3 (breaks the 80/20 split)
99f4e1c Add pytest for prepare_data and GitHub Actions CI
9c7188c Add Mage iris pipeline: load, prepare, train, save
5cca57a Add README, .gitignore and requirements
011bc4e Initial assignment setup


## 1. Install Mage in a Python 3.12 environment (same as in class)

In [ ]:
!pip install -q uv
!uv venv -q -p 3.12 /content/mage_env
!uv pip install -q --python /content/mage_env mage-ai scikit-learn
import os
os.environ['PATH'] = '/content/mage_env/bin:' + os.environ['PATH']
!mage --help | head -5

                                                                                
 Usage: mage [OPTIONS] COMMAND [ARGS]...                                        
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --install-completion          Install completion for the current shell.      │


## C1. Run the pipeline with tests (include this output in the report)

In [ ]:
!ls iris_project iris_project/pipelines/iris_classifier
!mage run iris_project iris_classifier --test

iris_project:
data_exporters	__init__.py	iris_mage_colab.ipynb  pipelines
data_loaders	io_config.yaml	metadata.yaml	       transformers

iris_project/pipelines/iris_classifier:
__init__.py  metadata.yaml
DBT library not installed.
INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running loop, using asyncio.run
Rows loaded: 150
--------------------------------------------------------------
1/1 tests passed.
Missing values: 0 | Classes: 3
--------------------------------------------------------------
1/1 tests passed.
Train rows: 120 | Test rows: 30
--------------------------------------------------------------
1/1 tests passed.
max_depth = 3 | Test accuracy = 1.0
--------------------------------------------------------------
1/1 tests passed.
Confusion matrix:
Saved 30 predictions to iris_predictions.csv
predicted   0  1   2
actual              
0          10  0   0
1           0  9   0
2           0  0  11
      metric  value
0   accuracy    1.0
1  precis

In [ ]:
import pandas as pd
pd.read_csv('iris_predictions.csv').head(10)

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,split,predicted
0,6.1,2.8,4.7,1.2,1,test,1
1,5.7,3.8,1.7,0.3,0,test,0
2,7.7,2.6,6.9,2.3,2,test,2
3,6.0,2.9,4.5,1.5,1,test,1
4,6.8,2.8,4.8,1.4,1,test,1
5,5.4,3.4,1.5,0.4,0,test,0
6,5.6,2.9,3.6,1.3,1,test,1
7,6.9,3.1,5.1,2.3,2,test,2
8,6.2,2.2,4.5,1.5,1,test,1
9,5.8,2.7,3.9,1.2,1,test,1


## C2. Tests as a gate: max_depth = 1 … 5
Before each run `iris_predictions.csv` is deleted. The table is filled in from the real output of each run.

In [ ]:
import subprocess, json, re, os, pandas as pd

rows = []
for d in [1, 2, 3, 4, 5]:
    if os.path.exists('iris_predictions.csv'):
        os.remove('iris_predictions.csv')
    r = subprocess.run(['mage', 'run', 'iris_project', 'iris_classifier', '--test',
                        '--runtime-vars', json.dumps({'max_depth': d})],
                       capture_output=True, text=True)
    out = r.stdout + r.stderr
    acc = re.search(r'Test accuracy = ([0-9.]+)', out)
    rows.append({
        'max_depth': d,
        'test accuracy': float(acc.group(1)) if acc else None,
        'test_accuracy passed?': 'Model accuracy is below 80%' not in out,
        'iris_predictions.csv created?': os.path.exists('iris_predictions.csv'),
    })
pd.DataFrame(rows)

,max_depth,test accuracy,test_accuracy passed?,iris_predictions.csv created?
0,1,0.633,False,False
1,2,0.967,True,True
2,3,1.000,True,True
3,4,1.000,True,True
4,5,1.000,True,True


In [ ]:
# Show the failing run in full (max_depth = 1): the assertion message and that save_results never ran
if os.path.exists('iris_predictions.csv'): os.remove('iris_predictions.csv')
!mage run iris_project iris_classifier --test --runtime-vars '{"max_depth": 1}' 2>&1 | grep -E "max_depth|Test accuracy|tests passed|Assertion|Saved|Failed"
print('csv exists:', os.path.exists('iris_predictions.csv'))

1/1 tests passed.
1/1 tests passed.
1/1 tests passed.
max_depth = 1 | Test accuracy = 0.633
AssertionError: Model accuracy is below 80%
0/1 tests passed.
│ ❱ 3385 │   │   │   │   │   raise Exception(f'Failed to pass tests for block  │
Exception: Failed to pass tests for block train_model
csv exists: False


## C3. The evaluation branch: `metadata.yaml`, confusion matrix and metrics

In [ ]:
!cat iris_project/pipelines/iris_classifier/metadata.yaml

blocks:
- name: load_iris
  uuid: load_iris
  type: data_loader
  language: python
  upstream_blocks: []
  downstream_blocks: [validate_iris]
- name: validate_iris
  uuid: validate_iris
  type: transformer
  language: python
  upstream_blocks: [load_iris]
  downstream_blocks: [prepare_data]
- name: prepare_data
  uuid: prepare_data
  type: transformer
  language: python
  upstream_blocks: [validate_iris]
  downstream_blocks: [train_model]
- name: train_model
  uuid: train_model
  type: transformer
  language: python
  upstream_blocks: [prepare_data]
  downstream_blocks: [save_results, evaluate_model]
- name: save_results
  uuid: save_results
  type: data_exporter
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: []
- name: evaluate_model
  uuid: evaluate_model
  type: transformer
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: []
name: iris_classifier
uuid: iris_classifier
type: python
variables:
  max_depth: 3


In [ ]:
# Normal run: the confusion matrix and the metrics table are printed by evaluate_model
!mage run iris_project iris_classifier --test 2>&1 | sed -n '/Confusion matrix/,$p'

Confusion matrix:
Saved 30 predictions to iris_predictions.csv
predicted   0  1   2
actual              
0          10  0   0
1           0  9   0
2           0  0  11
      metric  value
0   accuracy    1.0
1  precision    1.0
2     recall    1.0
3   f1_score    1.0
Pipeline run completed.


### Open the pipeline in the Mage UI (screenshot the graph with the branch)

In [ ]:
import subprocess, time, urllib.request

mage_server = subprocess.Popen(['/content/mage_env/bin/mage', 'start', 'iris_project'],
                               stdout=open('mage.log', 'w'), stderr=subprocess.STDOUT)
for i in range(60):
    try:
        urllib.request.urlopen('http://localhost:6789'); print('Mage is running'); break
    except Exception:
        time.sleep(2)
else:
    print('Mage did not start - check mage.log')

Mage is running


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
import subprocess, time, re
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:6789'],
                          stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(10)
link = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
print('Open Mage here:', link[0] if link else 'Not ready yet - run this cell again')
# In the UI: Pipelines -> iris_classifier -> the dependency tree shows train_model with TWO children.

Open Mage here: https://calvin-flooring-episodes-des.trycloudflare.com


## C4. My own block: `validate_iris` (transformer + @test, between load_iris and prepare_data)

In [ ]:
!cat iris_project/transformers/validate_iris.py
!mage run iris_project iris_classifier --test 2>&1 | grep -E "Missing values|tests passed"

if 'transformer' not in globals():
    try:
        from mage_ai.data_preparation.decorators import transformer
    except ImportError:
        def transformer(func):
            return func
if 'test' not in globals():
    try:
        from mage_ai.data_preparation.decorators import test
    except ImportError:
        def test(func):
            return func


# C4 (my own block): sits between load_iris and prepare_data.
# It passes the table through unchanged and reports how many missing values and classes it saw.
@transformer
def validate(df, *args, **kwargs):
    print('Missing values:', int(df.isna().sum().sum()), '| Classes:', df['target'].nunique())
    return df


# The gate for this block: no missing values and exactly 3 classes.
# If either fails, prepare_data and everything after it never run.
@test
def test_output(output, *args):
    assert output.isna().sum().sum() == 0, 'Iris must have no missing values'
    assert output['target'].nunique() == 3, 'Iris must have exactly 3